### Which products have the highest revenue?

In [26]:
from dotenv import load_dotenv
from databricks.connect import DatabricksSession
import os

load_dotenv() 

spark = DatabricksSession.builder \
    .host(os.getenv("DATABRICKS_HOST")) \
    .token(os.getenv("DATABRICKS_TOKEN")) \
    .serverless(True) \
    .getOrCreate()

# All audit log

In [17]:
spark.sql("""
    SELECT
    *
    FROM system.access.audit
    WHERE service_name = 'unityCatalog'
    ORDER BY event_time DESC
    LIMIT 20;
""").show()


+--------------------+------------+-------+--------------------+----------+-----------------+--------------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+
|          account_id|workspace_id|version|          event_time|event_date|source_ip_address|          user_agent|          session_id|       user_identity|service_name|         action_name|          request_id|      request_params|            response|  audit_level|            event_id|   identity_metadata|
+--------------------+------------+-------+--------------------+----------+-----------------+--------------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+
|17ecef56-5243-412...|           0|    2.0|2026-04-26 10:58:...|2026-0

# Who accessed the sensitive customers data?

In [28]:
spark.sql("""
    SELECT
        account_id,
        event_time,
        source_ip_address,
        user_identity.email,    
        action_name,
        request_params.table_full_name,
        request_params.operation
    FROM system.access.audit
    WHERE service_name = 'unityCatalog'
      AND request_params.table_full_name = 'workspace.gold.scd_customers'
""").toPandas()

,account_id,event_time,source_ip_address,email,action_name,table_full_name,operation
0,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-12 11:06:27.682,172.21.153.199,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ_WRITE
1,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-12 11:20:20.158,,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ
2,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-12 11:20:22.234,,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ
3,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-12 11:20:31.449,,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ_WRITE
4,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-12 11:20:31.189,,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ_WRITE
...,...,...,...,...,...,...,...
149,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-26 06:22:50.035,,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ
150,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-26 06:23:00.494,,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ
151,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-26 06:22:41.999,172.21.151.27,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ_WRITE
152,17ecef56-5243-4126-ac90-0e058e29a06d,2026-04-26 06:22:57.471,,yaobx04@gmail.com,generateTemporaryTableCredential,workspace.gold.scd_customers,READ


# Any denied access?

In [19]:
spark.sql("""
    SELECT
        event_time,
        user_identity.email,                     
        action_name,
        request_params.table_full_name,
        response.error_message
    FROM system.access.audit
    WHERE service_name = 'unityCatalog'
      AND response.status_code != 200
      AND request_params.table_full_name LIKE 'workspace.%'
    ORDER BY event_time DESC
""").toPandas()

,event_time,attempted_by,action_name,table_attempted,reason_denied


# Anyone change access recently?

In [20]:
spark.sql("""
    SELECT
        event_time,
        user_identity.email,                     
        action_name,
        request_params.securable_full_name,     
        request_params.changes              
    FROM system.access.audit
    WHERE service_name = 'unityCatalog'
      AND action_name IN (
          'updatePermissions',
          'grantPermissions',
          'revokePermissions'
      )
    ORDER BY event_time DESC
""").toPandas()

,event_time,email,action_name,securable_full_name,changes
